# Phase C v3 MVP: D 条件 shallow/deep 層窓 quick probe

## 目的
- `L_Ξ` が shallow な判別面を押しているのか、deep な構造面に届いているのかを最小コストで切る。
- 条件は **D のみ**、`baseline + λ=0.1`、`2-fold × 2 epochs` の quick probe に固定する。

## 実行行列
- `D × shallow × quick`
- `D × deep × quick`

## 必要ファイル
1. `phase_c_v3.py`
2. `phase_c_condition_D.jsonl`


In [ ]:
# Cell 1: 環境セットアップ
!nvidia-smi
!pip install -q transformers peft bitsandbytes accelerate datasets scipy scikit-learn


In [ ]:
# Cell 2: ファイルアップロード
from google.colab import files

print("2ファイルをアップロード:")
print("  1. phase_c_v3.py")
print("  2. phase_c_condition_D.jsonl")
uploaded = files.upload()
print(f"\n{len(uploaded)} files uploaded: {list(uploaded.keys())}")


In [ ]:
# Cell 3: shallow quick probe
!python phase_c_v3.py --condition D --data phase_c_condition_D.jsonl --quick --layer-window shallow --output phase_c_v3_D_shallow_quick.json


In [ ]:
# Cell 4: deep quick probe
!python phase_c_v3.py --condition D --data phase_c_condition_D.jsonl --quick --layer-window deep --output phase_c_v3_D_deep_quick.json


In [ ]:
# Cell 5: 比較 + 判定
import json

def load_pair(path):
    with open(path, encoding='utf-8') as f:
        payload = json.load(f)
    cond = payload['conditions'][0]
    base = cond['results']['baseline']
    lxi = cond['results']['lxi_0.1']
    return {
        'layer_window': cond['layer_window'],
        'layer_indices': cond['layer_indices'],
        'baseline_acc': base['mean_acc'],
        'baseline_partial_rho_ccl': base['mean_partial_rho_ccl'],
        'lxi_acc': lxi['mean_acc'],
        'lxi_partial_rho_ccl': lxi['mean_partial_rho_ccl'],
        'delta_acc': lxi['mean_acc'] - base['mean_acc'],
        'delta_partial_rho_ccl': lxi['mean_partial_rho_ccl'] - base['mean_partial_rho_ccl'],
    }

shallow = load_pair('phase_c_v3_D_shallow_quick.json')
deep = load_pair('phase_c_v3_D_deep_quick.json')
delta_delta = deep['delta_partial_rho_ccl'] - shallow['delta_partial_rho_ccl']

print('=' * 90)
print('Phase C v3 MVP — D shallow vs deep')
print('=' * 90)
for row in [shallow, deep]:
    print(f"layer_window={row['layer_window']}  layer_indices={row['layer_indices']}")
    print(f"  baseline: acc={row['baseline_acc']:.4f}  partial_ρ_ccl={row['baseline_partial_rho_ccl']:.4f}")
    print(f"  lxi_0.1 : acc={row['lxi_acc']:.4f}  partial_ρ_ccl={row['lxi_partial_rho_ccl']:.4f}")
    print(f"  Δacc={row['delta_acc']:+.4f}  Δpartial_ρ_ccl={row['delta_partial_rho_ccl']:+.4f}")
    print()

print(f"ΔΔpartial_ρ_ccl (deep - shallow) = {delta_delta:+.4f}")

if delta_delta >= 0.05 or (deep['delta_partial_rho_ccl'] * shallow['delta_partial_rho_ccl'] < 0):
    surprise = 'HIGH'
elif delta_delta >= 0.02:
    surprise = 'MED'
elif delta_delta >= 0.01:
    surprise = 'LOW'
else:
    surprise = 'ZERO'

if delta_delta >= 0.02 and deep['lxi_acc'] >= shallow['lxi_acc'] - 0.02:
    verdict = '支持'
elif delta_delta < 0.01 or (deep['delta_partial_rho_ccl'] < 0.01 and shallow['delta_partial_rho_ccl'] < 0.01) or (deep['lxi_acc'] < shallow['lxi_acc'] - 0.02):
    verdict = '棄却'
else:
    verdict = '不明'

print(f"Surprise={surprise}  判定={verdict}")


In [ ]:
# Cell 6: ダウンロード
files.download('phase_c_v3_D_shallow_quick.json')
files.download('phase_c_v3_D_deep_quick.json')
